# 07 — Drift Analysis & Data Stability

Analyzes the stability of both the input Population (Independent Variables) and the prediction Model (Dependent Variables) against a gold-standard benchmark.

## Generalization Standard
- Dynamically ingests `manifesto_config.pkl` to identify `Dev` (Benchmark) vs Evaluation datasets.
- Computes **Population Stability Index (PSI)** for model predictions and features.
- Calculates exhaustive 13-point percentile descriptions for continuous inputs.
- Calculates nulls and top 10 discrete frequencies for categorical inputs.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import pickle
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
print('Libraries and dependencies loaded.')

Libraries and dependencies loaded.


## 1. Directory Structure Setup

In [2]:
os.makedirs('../Outputs/drift_analysis/model_stability', exist_ok=True)
os.makedirs('../Outputs/drift_analysis/population_stability', exist_ok=True)
os.makedirs('../Outputs/drift_analysis/descriptive_statistics', exist_ok=True)

## 2. Load Central Configuration (Single Source of Truth)

In [3]:
with open('../Outputs/manifesto_config.pkl', 'rb') as f:
    manifesto = pickle.load(f)
dataset_configs = manifesto['dataset_configs']

with open('../Client_Input/Common/feature_engineering_artifacts.pkl', 'rb') as f:
    fe_artifacts = pickle.load(f)
feature_cols = fe_artifacts['feature_cols']

# Determine categoricals automatically vs numerical based on typical cardinality or predefined mapping if it exists.
# Here we'll rely on pd.api.types or explicit check.


## 3. Define Statistical Utilities

In [4]:
def calculate_psi(expected, actual, buckets=10):
    """Calculate Population Stability Index (PSI) between two arrays natively."""
    def build_buckets(x, num_buckets):
        breakpoints = np.arange(0, 100.1, 100 / num_buckets)
        return np.unique(np.percentile(x, breakpoints))
        
    # Remove NaNs
    expected = expected[~np.isnan(expected)]
    actual = actual[~np.isnan(actual)]
    
    if len(expected) == 0 or len(actual) == 0:
        return 0.0
        
    if len(np.unique(expected)) <= buckets:
        # Categorical or low cardinality discrete logic
        bins = np.unique(expected)
        expected_percents = []
        actual_percents = []
        for b in bins:
            expected_percents.append(sum(expected == b) / len(expected))
            actual_percents.append(sum(actual == b) / len(actual))
    else:
        # Continuous binning
        breakpoints = build_buckets(expected, buckets)
        # Ensure lower and upper bounds encompass entire range safely
        breakpoints[0] = -np.inf
        breakpoints[-1] = np.inf
        
        expected_counts = np.histogram(expected, breakpoints)[0]
        actual_counts = np.histogram(actual, breakpoints)[0]
        
        expected_percents = expected_counts / len(expected)
        actual_percents = actual_counts / len(actual)
        
    # Replace 0 with extremely small epsilon to prevent Inf / zero division error in log calculation
    eps = 1e-4
    expected_percents = np.where(expected_percents == 0, eps, expected_percents)
    actual_percents = np.where(actual_percents == 0, eps, actual_percents)
    
    psi_value = np.sum((expected_percents - actual_percents) * np.log(expected_percents / actual_percents))
    return psi_value

def extract_continuous_stats(series, feature_name):
    """Extract exactly 13 percentiles and descriptive stats for continuous variables."""
    s = pd.to_numeric(series, errors='coerce')
    n_count = len(s)
    null_count = s.isna().sum()
    valid_s = s.dropna()
    
    stats = {
        'Variable_Name': feature_name,
        'Type': 'Continuous',
        'Total_Count': n_count,
        'Null_Count': null_count,
        'Mean': valid_s.mean() if len(valid_s) else np.nan,
        'Std_Dev': valid_s.std() if len(valid_s) else np.nan
    }
    
    # Required Explicit Percentiles
    percentiles = [0.5, 1, 2.5, 5, 10, 25, 50, 75, 90, 95, 97.5, 99, 99.5]
    for p in percentiles:
        stats[f'p{p}'] = np.percentile(valid_s, p) if len(valid_s) else np.nan
        
    return stats

def extract_categorical_stats(series, feature_name):
    """Extract top 10 classes and frequencies for categorical variables."""
    n_count = len(series)
    null_count = series.isna().sum()
    counts = series.value_counts(dropna=True)
    
    # Handle top 10 dynamically. If fewer than 10 classes, pad with Nans.
    top_10 = counts.head(10)
    
    stats = {
        'Variable_Name': feature_name,
        'Type': 'Categorical',
        'Total_Count': n_count,
        'Null_Count': null_count
    }
    
    for i, (val, cnt) in enumerate(top_10.items(), 1):
        stats[f'Class_{i}_Value'] = val
        stats[f'Class_{i}_Count'] = cnt
        stats[f'Class_{i}_Pct'] = (cnt / n_count) * 100
        
    return stats


## 4. Compile Benchmark Data (Single Source of Truth Dataset)
We identify the distinct physical datasets. Dev is assigned as the universal benchmark.

In [5]:
# Reconstruct the unique physical datasets from the unified run schema
file_mappings = {}
for run in dataset_configs:
    ds_name = run['dataset_name']
    path = run['path']
    m_name = run['model_name']
    pred_col = run['pred_col']
    
    if path not in file_mappings:
        file_mappings[path] = {
            'dataset_name': ds_name,
            'models': {m_name: pred_col}
        }
    else:
        file_mappings[path]['models'][m_name] = pred_col

# Load mapping functionally
dfs = {path: pd.read_csv(path) for path in file_mappings.keys()}

# Partition physical DataFrames into Benchmark vs Evaluation targets
benchmark_dfs = {p: df for p, df in dfs.items() if file_mappings[p]['dataset_name'].lower() == 'dev'}
evaluation_dfs = {p: df for p, df in dfs.items() if file_mappings[p]['dataset_name'].lower() != 'dev'}

print(f"Compiled {len(benchmark_dfs)} distinct Benchmark blocks and {len(evaluation_dfs)} independent Evaluation blocks natively from config.")

Compiled 2 distinct Benchmark blocks and 2 independent Evaluation blocks natively from config.


## 5. Execute 1: Feature Statistics (Benchmark vs Evaluated Datasets)

In [6]:
def analyze_dataset_features(df, ds_name):
    all_stats = []
    # Differentiate Categorical structurally
    for col in df.columns:
        if col in feature_cols or col in ['Segment', 'TIME', 'QUANTITY']:
            # Simple heuristic: object/bool pd types or extremely low discrete integer cardinality
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]) or (pd.api.types.is_integer_dtype(df[col]) and df[col].nunique() < 15):
                st = extract_categorical_stats(df[col], col)
            else:
                st = extract_continuous_stats(df[col], col)
            
            st['Dataset'] = ds_name
            all_stats.append(st)
    return all_stats

master_descriptives = []

for path, d_info in file_mappings.items():
    print(f"Calculating descriptive metrics for Data Block: {d_info['dataset_name']}...")
    df_target = dfs[path]
    stats_list = analyze_dataset_features(df_target, d_info['dataset_name'])
    master_descriptives.extend(stats_list)

# Standardize and save schema
desc_df = pd.DataFrame(master_descriptives)
cat_df = desc_df[desc_df['Type'] == 'Categorical'].dropna(axis=1, how='all')
cont_df = desc_df[desc_df['Type'] == 'Continuous'].dropna(axis=1, how='all')

cat_df.to_csv('../Outputs/drift_analysis/descriptive_statistics/categorical_features_stats.csv', index=False)
cont_df.to_csv('../Outputs/drift_analysis/descriptive_statistics/continuous_features_stats.csv', index=False)
print("\u2705 Detailed Continuous and Categorical Percentile Distributions saved natively.")

Calculating descriptive metrics for Data Block: Dev...
Calculating descriptive metrics for Data Block: Dev...
Calculating descriptive metrics for Data Block: Validation...
Calculating descriptive metrics for Data Block: Validation...


✅ Detailed Continuous and Categorical Percentile Distributions saved natively.


## 6. Execute 2 & 3: Model Stability & Feature Population Stability (PSI)

In [7]:
psi_model_records = []
psi_feature_records = []

# We must pair the evaluation sets to their corresponding pipeline architectures
# Since XGBoost Dev maps to XGBoost Val natively, we join on the available prediction models.
for eval_path, eval_df in evaluation_dfs.items():
    eval_ds_name = file_mappings[eval_path]['dataset_name']
    eval_models = file_mappings[eval_path]['models']
    
    # Find a matching benchmark path (that shares the exact same models!)
    # e.g. dev_xgboost matches val_xgboost because they both contain `xgb_predicted_quantity`
    matching_bench_path = None
    for bench_path in benchmark_dfs.keys():
        if set(file_mappings[bench_path]['models'].keys()) == set(eval_models.keys()):
            matching_bench_path = bench_path
            break
            
    if not matching_bench_path:
        print(f"  [Warning] Could not pair Evaluation block {eval_ds_name} with an exact matching feature Dev benchmark. Skip PSI.")
        continue
        
    bench_df = benchmark_dfs[matching_bench_path]
    bench_ds_name = file_mappings[matching_bench_path]['dataset_name']
    
    print(f"\nEvaluating PSI between benchmark ({bench_ds_name}) and ({eval_ds_name})")
    
    # 1. Prediction Model Stability (PSI Model Core)
    for m_name, pred_col in eval_models.items():
        if pred_col in bench_df.columns and pred_col in eval_df.columns:
            model_psi = calculate_psi(bench_df[pred_col].values, eval_df[pred_col].values)
            psi_model_records.append({
                'Benchmark': bench_ds_name,
                'Evaluating_Dataset': eval_ds_name,
                'Model': m_name,
                'Model_Stability_PSI': model_psi,
                'Status': 'Stable' if model_psi < 0.1 else ('Monitor' if model_psi < 0.25 else 'Significant Drift')
            })
            print(f"  -> Model PSI ({m_name}): {model_psi:.4f}")

    # 2. Input Population Feature Stability (PSI Inputs)
    for f in feature_cols:
        if f in bench_df.columns and f in eval_df.columns:
            feat_psi = calculate_psi(bench_df[f].values, eval_df[f].values)
            psi_feature_records.append({
                'Benchmark': bench_ds_name,
                'Evaluating_Dataset': eval_ds_name,
                'Assoc_Models': " | ".join(list(eval_models.keys())),
                'Feature': f,
                'Feature_PSI': feat_psi,
                'Status': 'Stable' if feat_psi < 0.1 else ('Monitor' if feat_psi < 0.25 else 'Significant Drift')
            })

# Exports
pd.DataFrame(psi_model_records).to_csv('../Outputs/drift_analysis/model_stability/model_stability_psi.csv', index=False)
pd.DataFrame(psi_feature_records).to_csv('../Outputs/drift_analysis/population_stability/feature_population_psi.csv', index=False)
print("\n\u2705 Feature mapping and prediction model PSIs completely exported dynamically!")



Evaluating PSI between benchmark (Dev) and (Validation)
  -> Model PSI (XGBoost): 0.2005
  -> Model PSI (LightGBM): 0.1929

Evaluating PSI between benchmark (Dev) and (Validation)
  -> Model PSI (ARDL Base): 0.1806
  -> Model PSI (ARDL Adjusted): 0.1868



✅ Feature mapping and prediction model PSIs completely exported dynamically!
